In [1]:
#  Imports and Spark Session Initialization
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, to_date, count, avg, month, year

# Initialize Spark Session with MongoDB Connector
spark = SparkSession.builder \
    .appName("AmazonReviewsETL") \
    .config("spark.mongodb.write.connection.uri", "mongodb://localhost:27017/amazon_db") \
    .config("spark.jars.packages", "org.mongodb.spark:mongo-spark-connector_2.12:10.4.0") \
    .getOrCreate()

# Suppress INFO logs to keep the notebook output clean (show only WARN/ERROR)
spark.sparkContext.setLogLevel("WARN")

print("Spark Session successfully initialized!")

:: loading settings :: url = jar:file:/Users/macpro/Documents/MyProject/venv/lib/python3.12/site-packages/pyspark/jars/ivy-2.5.1.jar!/org/apache/ivy/core/settings/ivysettings.xml


Ivy Default Cache set to: /Users/macpro/.ivy2/cache
The jars for the packages stored in: /Users/macpro/.ivy2/jars
org.mongodb.spark#mongo-spark-connector_2.12 added as a dependency
:: resolving dependencies :: org.apache.spark#spark-submit-parent-217393b7-bb73-476d-a64d-88274c6e7104;1.0
	confs: [default]
	found org.mongodb.spark#mongo-spark-connector_2.12;10.4.0 in central
	found org.mongodb#mongodb-driver-sync;5.1.4 in central
	[5.1.4] org.mongodb#mongodb-driver-sync;[5.1.1,5.1.99)
	found org.mongodb#bson;5.1.4 in central
	found org.mongodb#mongodb-driver-core;5.1.4 in central
	found org.mongodb#bson-record-codec;5.1.4 in central
downloading https://repo1.maven.org/maven2/org/mongodb/spark/mongo-spark-connector_2.12/10.4.0/mongo-spark-connector_2.12-10.4.0.jar ...
	[SUCCESSFUL ] org.mongodb.spark#mongo-spark-connector_2.12;10.4.0!mongo-spark-connector_2.12.jar (407ms)
downloading https://repo1.maven.org/maven2/org/mongodb/mongodb-driver-sync/5.1.4/mongodb-driver-sync-5.1.4.jar ...
	[S

Spark Session successfully initialized!


In [2]:
# - Data Ingestion
# Path to the downloaded CSV file on the local machine
file_path = "/Users/macpro/Downloads/amazon_reviews.csv"

# Load the CSV file into a Spark DataFrame
df_raw = spark.read.csv(file_path, header=True, inferSchema=True)

print(f"Total rows before cleaning: {df_raw.count()}")

# Display the inferred schema to verify column data types
df_raw.printSchema()

Total rows before cleaning: 396000
root
 |-- marketplace: string (nullable = true)
 |-- customer_id: integer (nullable = true)
 |-- review_id: string (nullable = true)
 |-- product_id: string (nullable = true)
 |-- product_parent: integer (nullable = true)
 |-- product_title: string (nullable = true)
 |-- product_category: string (nullable = true)
 |-- star_rating: string (nullable = true)
 |-- helpful_votes: string (nullable = true)
 |-- total_votes: string (nullable = true)
 |-- vine: string (nullable = true)
 |-- verified_purchase: string (nullable = true)
 |-- review_headline: string (nullable = true)
 |-- review_body: string (nullable = true)
 |-- review_date: string (nullable = true)



In [3]:
#  Data Cleaning and Filtering
df_clean = (
    df_raw
    # 1. Remove rows with null values in critical columns
    .dropna(subset=["review_id", "product_id", "star_rating", "review_date"])

    # 2. Convert 'review_date' string to a Spark DateType format (yyyy-MM-dd)
    .withColumn("review_date", to_date(col("review_date"), "yyyy-MM-dd"))

    # 3. Filter reviews to include only verified purchases
    # Using .isin() to handle potential variations in the CSV (e.g., 1, "1", "Y")
    .filter(col("verified_purchase").isin([1, "1", "Y", "y"]))
)

# Cache the cleaned DataFrame in memory
# This optimizes performance since we will use it for 3 separate aggregations below
df_clean.cache()

print(f"Total rows after cleaning: {df_clean.count()}")

# Show the first 5 rows to visually verify the clean data
df_clean.show(5)

Total rows after cleaning: 46792
+-----------+-----------+--------------+----------+--------------+--------------------+----------------+-----------+-------------+-----------+----+-----------------+--------------------+--------------------+-----------+
|marketplace|customer_id|     review_id|product_id|product_parent|       product_title|product_category|star_rating|helpful_votes|total_votes|vine|verified_purchase|     review_headline|         review_body|review_date|
+-----------+-----------+--------------+----------+--------------+--------------------+----------------+-----------+-------------+-----------+----+-----------------+--------------------+--------------------+-----------+
|         US|   48541186|R1VE0FQQ0QTQJN|0300108834|     915891133|A Little History ...|           Books|          5|           16|         20|   0|                1|Simple, entertain...|Never been much f...| 2005-10-14|
|         US|   52253037|R21SYDQ70ILUC0|1580085695|     586052746|Furry Logic: A Gu...|

In [4]:
# Aggregation 1 - Product Stats
# Calculate and store the total number of reviews and average star rating per product
df_product_stats = df_clean.groupBy("product_id").agg(
    count("review_id").alias("total_reviews"),
    avg("star_rating").alias("avg_rating")
)

print("--- Product Statistics ---")
df_product_stats.show(5)

--- Product Statistics ---
+----------+-------------+----------+
|product_id|total_reviews|avg_rating|
+----------+-------------+----------+
|0842329129|            3|       1.0|
|0826472621|            1|       5.0|
|0762404515|            1|       1.0|
|1588513521|            2|       5.0|
|0972437746|            1|       1.0|
+----------+-------------+----------+
only showing top 5 rows



In [5]:
# Aggregation 2 - Customer Activity
# Determine the total number of verified reviews submitted by each customer
df_customer_activity = df_clean.groupBy("customer_id").agg(
    count("review_id").alias("verified_reviews_count")
)

print("--- Customer Activity ---")
df_customer_activity.show(5)

--- Customer Activity ---
+-----------+----------------------+
|customer_id|verified_reviews_count|
+-----------+----------------------+
|   15596972|                     1|
|   14544320|                     1|
|   52833886|                     1|
|   51088740|                     1|
|   27851871|                     2|
+-----------+----------------------+
only showing top 5 rows



In [6]:
# Aggregation 3 - Product Monthly Trends
# Aggregate the monthly number of reviews per product
# First, extract the 'month' and 'year' from the 'review_date' column
df_product_trends = (
    df_clean.withColumn("month", month("review_date"))
            .withColumn("year", year("review_date"))
            .groupBy("product_id", "year", "month")
            .agg(count("review_id").alias("review_count"))
)

print("--- Product Monthly Trends ---")
df_product_trends.show(5)

--- Product Monthly Trends ---
+----------+----+-----+------------+
|product_id|year|month|review_count|
+----------+----+-----+------------+
|0486265250|2005|   10|           1|
|0618338055|2005|   10|           1|
|0515128635|2005|   10|           1|
|0375726810|2005|   10|           1|
|0441012906|2005|   10|           1|
+----------+----+-----+------------+
only showing top 5 rows



In [ ]:
#  Load Process (Saving results to MongoDB)
print("Starting data load to MongoDB...")

# 1. Save product stats to the 'product_stats' collection
df_product_stats.write.format("mongodb") \
    .mode("overwrite") \
    .option("collection", "product_stats") \
    .save()
print("1/3: 'product_stats' collection saved successfully.")

# 2. Save customer activity to the 'customer_activity' collection
df_customer_activity.write.format("mongodb") \
    .mode("overwrite") \
    .option("collection", "customer_activity") \
    .save()
print("2/3: 'customer_activity' collection saved successfully.")

# 3. Save monthly trends to the 'product_trends' collection
df_product_trends.write.format("mongodb") \
    .mode("overwrite") \
    .option("collection", "product_trends") \
    .save()
print("3/3: 'product_trends' collection saved successfully.")

print("ETL process completed! You can now check MongoDB Compass.")